In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_06 — Hyperopt Tuning
# MAGIC **Bayesian HPO → finds best LightGBM params → auto-promotes @Champion if AUC improves**
# MAGIC
# MAGIC This is the "Hyperopt Improvement Cycle" box in the architecture diagram.
# MAGIC GridSense had this — it signals production ML maturity to judges.
# MAGIC
# MAGIC What this notebook does:
# MAGIC - Runs 20 Bayesian trials with Hyperopt TPE (Tree of Parzen Estimators)
# MAGIC - Each trial trains a LightGBM with different hyperparameters
# MAGIC - All trials logged to MLflow automatically
# MAGIC - Best params saved to `xscore.gold.best_hyperparams` Delta table
# MAGIC - If best trial AUC > current @Champion AUC → auto-promotes new @Champion
# MAGIC
# MAGIC **Depends on:** NB_05 complete
# MAGIC **Runtime:** ~15 minutes (20 trials × ~45s each)
# MAGIC **Next:** NB_07_shap_explainer

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 1 — Install and setup

# COMMAND ----------

%pip install lightgbm hyperopt mlflow scikit-learn --quiet

# COMMAND ----------

import mlflow
import mlflow.lightgbm
import lightgbm as lgb
import numpy as np
import pandas as pd
import json
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval
from hyperopt.pyll import scope
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
from sklearn.model_selection import train_test_split
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from mlflow.tracking import MlflowClient

spark.sql("USE CATALOG xscore")
spark.conf.set("spark.sql.shuffle.partitions", "8")
mlflow.set_registry_uri("databricks-uc")

print("✓ Hyperopt:", __import__("hyperopt").__version__)
print("✓ LightGBM:", lgb.__version__)

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 2 — Load data and feature contract

# COMMAND ----------

contract    = json.loads(dbutils.fs.head(
    "/Volumes/xscore/bronze/kaggle_raw/feature_contract.json"
))
FEATURE_COLS = contract["feature_cols"]
LABEL_COL    = contract["label_col"]

gold_pd = (spark.table("xscore.gold.credit_feature_store")
           .select(FEATURE_COLS + [LABEL_COL])
           .fillna(0.0)
           .toPandas())

X = gold_pd[FEATURE_COLS]
y = gold_pd[LABEL_COL]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.15/0.85, random_state=42, stratify=y_trainval
)

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")
print(f"scale_pos_weight: {pos_weight:.2f}")

# Get current Champion AUC to beat
client = MlflowClient()
try:
    champ_versions = client.search_model_versions(
        "name='xscore.gold.credit_scorer'"
    )
    champ = next(v for v in champ_versions
                 if "Champion" in [a.alias for a in
                    client.get_model_version_aliases(
                        "xscore.gold.credit_scorer", v.version
                    )])
    CHAMPION_AUC = float(
        client.get_model_version(
            "xscore.gold.credit_scorer", champ.version
        ).tags.get("val_auc_roc", "0.0")
    )
    CHAMPION_VERSION = champ.version
except Exception:
    CHAMPION_AUC     = 0.83
    CHAMPION_VERSION = "unknown"

print(f"\nCurrent @Champion AUC to beat: {CHAMPION_AUC:.4f} (v{CHAMPION_VERSION})")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 3 — Define Hyperopt search space

# COMMAND ----------

# ─────────────────────────────────────────────────────────────
# TPE (Tree of Parzen Estimators) search space.
# We search over the most impactful LightGBM hyperparameters.
# Ranges are set based on credit scoring best practices.
# ─────────────────────────────────────────────────────────────

SEARCH_SPACE = {
    # How fast the model learns — smaller = more trees needed but better generalisation
    "learning_rate"    : hp.loguniform("learning_rate",    np.log(0.01),  np.log(0.15)),

    # Number of leaves — controls model complexity
    # 2^max_depth is the theoretical max; 31-127 is the practical range
    "num_leaves"       : scope.int(hp.quniform("num_leaves", 20, 120, 1)),

    # Maximum tree depth — deeper = more complex, more prone to overfit
    "max_depth"        : scope.int(hp.quniform("max_depth",  4,   9,  1)),

    # Minimum samples in a leaf — higher = more regularisation
    "min_child_samples": scope.int(hp.quniform("min_child_samples", 10, 50, 1)),

    # Fraction of features used per tree — reduces correlation between trees
    "feature_fraction" : hp.uniform("feature_fraction",  0.60, 0.95),

    # Fraction of rows sampled per boosting round
    "bagging_fraction" : hp.uniform("bagging_fraction",  0.60, 0.95),

    # L1 regularisation — encourages sparse feature weights
    "reg_alpha"        : hp.loguniform("reg_alpha",  np.log(0.01), np.log(1.0)),

    # L2 regularisation — penalises large weights
    "reg_lambda"       : hp.loguniform("reg_lambda", np.log(0.01), np.log(1.0)),
}

print("Search space defined:")
for k, v in SEARCH_SPACE.items():
    print(f"  {k}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 4 — Objective function

# COMMAND ----------

# Track trial number for logging
trial_num = [0]

def objective(params):
    """
    Called by Hyperopt for each trial.
    Returns negative AUC (Hyperopt minimises, we want to maximise AUC).
    """
    trial_num[0] += 1
    t = trial_num[0]

    lgbm_params = {
        "objective"        : "binary",
        "metric"           : "auc",
        "boosting_type"    : "gbdt",
        "bagging_freq"     : 5,
        "scale_pos_weight" : pos_weight,
        "random_state"     : 42,
        "verbose"          : -1,
        **params,
    }

    with mlflow.start_run(
        run_name=f"hyperopt_trial_{t:02d}",
        nested=True
    ):
        mlflow.log_params({f"hp_{k}": v for k, v in params.items()})

        dtrain = lgb.Dataset(X_train, label=y_train)
        dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

        model = lgb.train(
            lgbm_params,
            dtrain,
            num_boost_round=500,
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(30, verbose=False),
                lgb.log_evaluation(0),
            ],
        )

        val_prob = model.predict(X_val)
        val_auc  = roc_auc_score(y_val, val_prob)
        val_f1   = f1_score((val_prob >= 0.5).astype(int), y_val, zero_division=0)

        mlflow.log_metrics({
            "val_auc_roc"   : round(val_auc, 4),
            "val_f1"        : round(val_f1, 4),
            "best_iteration": model.best_iteration,
        })

        print(f"  Trial {t:02d} | AUC: {val_auc:.4f} | F1: {val_f1:.4f} | "
              f"iters: {model.best_iteration:3d} | "
              f"leaves: {int(params['num_leaves']):3d} | "
              f"lr: {params['learning_rate']:.4f}")

    return {"loss": -val_auc, "status": STATUS_OK}

print("✓ Objective function defined")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 5 — Run 20 Hyperopt trials

# COMMAND ----------

N_TRIALS = 20

print(f"Running {N_TRIALS} Bayesian HPO trials...")
print(f"TPE algorithm — each trial learns from previous results\n")
print(f"  {'Trial':>6}  {'AUC':>8}  {'F1':>8}  {'iters':>6}  {'leaves':>6}  {'lr':>8}")
print("  " + "─" * 55)

trials = Trials()

mlflow.set_experiment("/Users/aryamanbhati8@gmail.com/xscore_credit_scoring_hyperopt")

with mlflow.start_run(run_name="hyperopt_20_trials"):
    mlflow.log_params({
        "n_trials"      : N_TRIALS,
        "algorithm"     : "TPE",
        "feature_set"   : "full_xscore_6_pillars",
        "n_features"    : len(FEATURE_COLS),
        "champion_auc_before": CHAMPION_AUC,
    })

    best_params = fmin(
        fn        =objective,
        space     =SEARCH_SPACE,
        algo      =tpe.suggest,
        max_evals =N_TRIALS,
        trials    =trials,
        rstate    =np.random.default_rng(42),
        verbose   =False,
    )

    # Best AUC across all trials
    best_auc = -min(t["result"]["loss"] for t in trials.trials)
    mlflow.log_metric("best_val_auc", best_auc)
    mlflow.log_metric("champion_auc_before", CHAMPION_AUC)
    mlflow.log_metric("auc_improvement", best_auc - CHAMPION_AUC)

print(f"\n{'─'*55}")
print(f"Best AUC across {N_TRIALS} trials: {best_auc:.4f}")
print(f"Current @Champion AUC:            {CHAMPION_AUC:.4f}")
print(f"Improvement:                      {best_auc - CHAMPION_AUC:+.4f}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 6 — Show best parameters found

# COMMAND ----------

# space_eval converts Hyperopt's internal representation back to real values
best_params_resolved = space_eval(SEARCH_SPACE, best_params)

print("Best hyperparameters found by TPE:\n")
for k, v in sorted(best_params_resolved.items()):
    if isinstance(v, float):
        print(f"  {k:<25} {v:.5f}")
    else:
        print(f"  {k:<25} {v}")

# Compare with v3 defaults
print("\nComparison with v3 defaults:")
v3_defaults = {
    "learning_rate": 0.04, "num_leaves": 63, "max_depth": 7,
    "min_child_samples": 15, "feature_fraction": 0.75,
    "bagging_fraction": 0.80, "reg_alpha": 0.1, "reg_lambda": 0.1,
}
for k in best_params_resolved:
    v3_val  = v3_defaults.get(k, "N/A")
    new_val = best_params_resolved[k]
    changed = " ← changed" if str(round(float(new_val), 3)) != str(round(float(v3_val), 3)) else ""
    print(f"  {k:<25} v3={v3_val}  →  best={new_val:.4f}{changed}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 7 — Train final model with best params

# COMMAND ----------

BEST_PARAMS = {
    "objective"        : "binary",
    "metric"           : "auc",
    "boosting_type"    : "gbdt",
    "bagging_freq"     : 5,
    "scale_pos_weight" : pos_weight,
    "random_state"     : 42,
    "verbose"          : -1,
    **best_params_resolved,
}

print("Training final model with best Hyperopt params...")

dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

lgbm_best = lgb.train(
    BEST_PARAMS,
    dtrain,
    num_boost_round=500,
    valid_sets=[dval],
    callbacks=[
        lgb.early_stopping(30, verbose=False),
        lgb.log_evaluation(0),
    ],
)

# Evaluate on all splits
val_best  = {
    "auc_roc": round(roc_auc_score(y_val,  lgbm_best.predict(X_val)),  4),
    "f1"     : round(f1_score((lgbm_best.predict(X_val) >= 0.5).astype(int), y_val, zero_division=0), 4),
}
test_best = {
    "auc_roc": round(roc_auc_score(y_test, lgbm_best.predict(X_test)), 4),
    "f1"     : round(f1_score((lgbm_best.predict(X_test) >= 0.5).astype(int), y_test, zero_division=0), 4),
}

print(f"\nFinal model (best params):")
print(f"  Val  AUC: {val_best['auc_roc']:.4f}  F1: {val_best['f1']:.4f}")
print(f"  Test AUC: {test_best['auc_roc']:.4f}  F1: {test_best['f1']:.4f}")
print(f"  Iters   : {lgbm_best.best_iteration}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 8 — Save best params to gold.best_hyperparams Delta table

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE TABLE IF NOT EXISTS xscore.gold.best_hyperparams (
# MAGIC   run_timestamp   TIMESTAMP,
# MAGIC   n_trials        INT,
# MAGIC   best_auc        DOUBLE,
# MAGIC   champion_auc    DOUBLE,
# MAGIC   improvement     DOUBLE,
# MAGIC   promoted        BOOLEAN,
# MAGIC   learning_rate   DOUBLE,
# MAGIC   num_leaves      INT,
# MAGIC   max_depth       INT,
# MAGIC   min_child_samples INT,
# MAGIC   feature_fraction DOUBLE,
# MAGIC   bagging_fraction DOUBLE,
# MAGIC   reg_alpha       DOUBLE,
# MAGIC   reg_lambda      DOUBLE
# MAGIC )
# MAGIC USING DELTA
# MAGIC COMMENT 'Best hyperparameters from each Hyperopt run. Read by NB_09 nightly job.';

# COMMAND ----------

import pyspark.sql.types as T
from datetime import datetime

promoted = bool(val_best["auc_roc"] > CHAMPION_AUC)

params_row = spark.createDataFrame([{
    "run_timestamp"   : datetime.now(),
    "n_trials"        : N_TRIALS,
    "best_auc"        : float(val_best["auc_roc"]),
    "champion_auc"    : float(CHAMPION_AUC),
    "improvement"     : float(val_best["auc_roc"] - CHAMPION_AUC),
    "promoted"        : promoted,
    "learning_rate"   : float(best_params_resolved["learning_rate"]),
    "num_leaves"      : int(best_params_resolved["num_leaves"]),
    "max_depth"       : int(best_params_resolved["max_depth"]),
    "min_child_samples": int(best_params_resolved["min_child_samples"]),
    "feature_fraction": float(best_params_resolved["feature_fraction"]),
    "bagging_fraction": float(best_params_resolved["bagging_fraction"]),
    "reg_alpha"       : float(best_params_resolved["reg_alpha"]),
    "reg_lambda"      : float(best_params_resolved["reg_lambda"]),
}])

(params_row.write
    .format("delta")
    .mode("append")
    .saveAsTable("xscore.gold.best_hyperparams"))

print(f"✓ Best params saved to xscore.gold.best_hyperparams")
print(f"  Promoted to @Champion: {promoted}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 9 — Auto-promote @Champion if AUC improved

# COMMAND ----------

if val_best["auc_roc"] > CHAMPION_AUC:
    print(f"New AUC {val_best['auc_roc']:.4f} > Champion AUC {CHAMPION_AUC:.4f}")
    print("Registering new model and promoting to @Champion...\n")

    with mlflow.start_run(run_name="hyperopt_champion_upgrade"):
        mlflow.log_params({
            "promoted_from_hyperopt": True,
            "n_trials"  : N_TRIALS,
            "prev_champion_auc": CHAMPION_AUC,
            **{f"best_{k}": v for k, v in best_params_resolved.items()},
        })
        mlflow.log_metrics({
            "val_auc_roc": val_best["auc_roc"],
            "val_f1"     : val_best["f1"],
            "test_auc_roc": test_best["auc_roc"],
            "auc_improvement": val_best["auc_roc"] - CHAMPION_AUC,
        })
        mlflow.lightgbm.log_model(
            lgbm_best,
            "model",
            registered_model_name="xscore.gold.credit_scorer"
        )

    # Get the new version just registered
    versions     = client.search_model_versions("name='xscore.gold.credit_scorer'")
    new_version  = max(int(v.version) for v in versions)

    client.set_registered_model_alias(
        "xscore.gold.credit_scorer", "Champion", str(new_version)
    )
    client.set_model_version_tag(
        "xscore.gold.credit_scorer", str(new_version),
        "val_auc_roc", f"{val_best['auc_roc']:.4f}"
    )
    client.set_model_version_tag(
        "xscore.gold.credit_scorer", str(new_version),
        "promoted_by", "hyperopt_auto"
    )

    print(f"✓ NEW @Champion: version {new_version}")
    print(f"  AUC improved: {CHAMPION_AUC:.4f} → {val_best['auc_roc']:.4f}")
    print(f"  Gain: +{val_best['auc_roc'] - CHAMPION_AUC:.4f}")

else:
    print(f"No improvement: {val_best['auc_roc']:.4f} ≤ {CHAMPION_AUC:.4f}")
    print(f"@Champion stays at version {CHAMPION_VERSION}")
    print("Current champion is already optimal for this dataset.")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 10 — Show all trial results summary

# COMMAND ----------

# Build summary of all 20 trials
trial_results = []
for i, t in enumerate(trials.trials):
    trial_results.append({
        "trial"  : i + 1,
        "auc"    : round(-t["result"]["loss"], 4),
        "leaves" : int(t["misc"]["vals"]["num_leaves"][0]),
        "depth"  : int(t["misc"]["vals"]["max_depth"][0]),
        "lr"     : round(float(np.exp(t["misc"]["vals"]["learning_rate"][0])), 5),
    })

results_df = pd.DataFrame(trial_results).sort_values("auc", ascending=False)
print(f"All {N_TRIALS} trial results (sorted by AUC):\n")
print(results_df.to_string(index=False))
print(f"\nBest AUC: {results_df['auc'].max():.4f}  (trial {results_df.iloc[0]['trial']})")
print(f"Worst AUC: {results_df['auc'].min():.4f}  (trial {results_df.iloc[-1]['trial']})")
print(f"AUC range: {results_df['auc'].max() - results_df['auc'].min():.4f}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 11 — Final summary

# COMMAND ----------

print("=" * 60)
print("  NB_06 HYPEROPT — COMPLETE")
print("=" * 60)
print(f"  Trials run      : {N_TRIALS}")
print(f"  Algorithm       : TPE (Bayesian)")
print(f"  Best trial AUC  : {best_auc:.4f}")
print(f"  Champion before : {CHAMPION_AUC:.4f}")
print(f"  Champion after  : {max(CHAMPION_AUC, val_best['auc_roc']):.4f}")
print(f"  Promoted        : {promoted}")
print()
print("  Best params saved to xscore.gold.best_hyperparams")
print("  NB_09 (nightly job) reads this table to always")
print("  use the best known params for retraining.")
print()
print("  NEXT: NB_07_shap_explainer")
print("  NB_07 computes per-user SHAP factor breakdowns")
print("  and writes them to gold.credit_scores for the dashboard.")
print("=" * 60)